[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chihuahualee828/TorchCode/blob/master/solutions/42_mini_llm_solution.ipynb)

# Solution 42: Decoder-only LLM

Build a complete, trainable **Llama-style MiniLLM** using `LLMConfig`. Assignment 43 imports your model for training and chat. Design the implementation yourself.

## Requirements

- Token embeddings, exactly `config.num_layers` decoder layers, a final RMSNorm before the vocabulary head, and tied embedding/output weights.
- **Pre-norm:** within each layer, causal grouped-query attention comes first, then SwiGLU. Each sublayer normalizes its input with its own RMSNorm before computation and adds its output to the unnormalized residual stream. There is no post-residual norm within the layer.
- Use `nn.RMSNorm` with `config.eps` (PyTorch 2.4+), bias-free projections and no dropout.
- RoPE on Q/K: full head width, adjacent pairs, positions starting at zero, base `config.rope_base`, and frequency `base ** (-2*i/head_dim)` for pair `i`. Consecutive query-head groups share a KV head. No learned position embeddings.

**Allowed:** basic PyTorch layers/functions, normalization, activations, autograd and compatible built-in RoPE operations. A custom RoPE class/function is **not required**. Implement attention and decoder/model assembly yourself; no prebuilt attention/Transformer models, pretrained weights or judge oracle calls.

PyTorch documents [`torch.onnx.ops.rotary_embedding`](https://docs.pytorch.org/docs/2.14/onnx_ops.html#torch.onnx.ops.rotary_embedding), but its direct call fails backward in the tested environment. Any RoPE implementation you choose must preserve gradients and match the specified convention. The solution uses differentiable tensor operations.

## Configuration

Architecture choices above are fixed; dimensions below come from `LLMConfig`.
Use the passed configuration rather than hardcoding defaults: the judge varies them.

| Field | Meaning | Default |
|---|---|---|
| `num_layers` | Number of decoder layers | 2 |
| `d_model` | Embedding and residual width | 64 |
| `num_heads` | Query heads | 4 |
| `num_kv_heads` | Key/value heads | 2 |
| `hidden_dim` | SwiGLU intermediate width | 128 |
| `vocab_size` | Input/output vocabulary size | 260 |
| `max_seq_len` | Maximum input length | 256 |
| `rope_base` | RoPE frequency base | 10000.0 |
| `eps` | RMSNorm epsilon | 1e-5 |

`LLMConfig` validates positive dimensions, head divisibility and even head width.
Pre-norm, SwiGLU, RoPE and weight tying are requirements, not config switches.

## Interface and grading contract

`MiniLLM(config)` must be an `nn.Module`. `forward(input_ids)` takes integer IDs `(B,T)` and returns raw logits `(B,T,vocab_size)`. Reject non-2D inputs and lengths outside `1..max_seq_len` with `ValueError`. Preserve dtype, device and gradients; support right padding.

Use these module names to let the judge load fixed weights:

| Model | Each decoder layer |
|---|---|
| `embed_tokens`, `layers` (ModuleList), `norm`, `lm_head` | `attn_norm`, `ffn_norm`, `q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj` |

Tie `lm_head.weight` to the same Parameter as `embed_tokens.weight`. The feed-forward width is `config.hidden_dim`; attention head width is `d_model // num_heads`.

## Submit

Run the implementation cell, reload the exported module, then run `check("mini_llm")`. Fixed CPU checks cover logits, parameter gradients, causality, head configurations, context limits and serialization. Results are reproducible in the same environment.

The exported `mini_llm_reference.py` must be available to assignment 43 solution.


In [ ]:
# Use this repository version: these new tasks may not be on PyPI yet.
# Local: install with `pip install -e /path/to/TorchCode` before launching Jupyter.
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q git+https://github.com/chihuahualee828/TorchCode.git@master')
except ImportError:
    pass

import torch
from torch_judge import check, hint
from torch_judge.capstone import LLMConfig, deterministic

if not hasattr(torch.nn, "RMSNorm"):
    raise ImportError("Assignment 42 requires PyTorch 2.4+. Install torch>=2.4 and restart the kernel.")


In [ ]:
%%writefile mini_llm_reference.py
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch_judge.capstone import LLMConfig


def apply_rope(x, base):
    # x: (batch, heads, sequence, head_dim), adjacent-pair convention.
    dim = x.shape[-1]
    pos = torch.arange(x.shape[-2], device=x.device, dtype=x.dtype)
    inv = base ** (-torch.arange(0, dim, 2, device=x.device, dtype=x.dtype) / dim)
    angles = pos[:, None] * inv[None, :]
    a, b = x[..., 0::2], x[..., 1::2]
    return torch.stack((a * angles.cos() - b * angles.sin(),
                        a * angles.sin() + b * angles.cos()), -1).flatten(-2)


class DecoderLayer(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        d, h = config.d_model, config.d_model // config.num_heads
        self.attn_norm = nn.RMSNorm(d, eps=config.eps)
        self.ffn_norm = nn.RMSNorm(d, eps=config.eps)
        self.q_proj = nn.Linear(d, d, bias=False)
        self.k_proj = nn.Linear(d, config.num_kv_heads * h, bias=False)
        self.v_proj = nn.Linear(d, config.num_kv_heads * h, bias=False)
        self.o_proj = nn.Linear(d, d, bias=False)
        self.gate_proj = nn.Linear(d, config.hidden_dim, bias=False)
        self.up_proj = nn.Linear(d, config.hidden_dim, bias=False)
        self.down_proj = nn.Linear(config.hidden_dim, d, bias=False)

    def forward(self, x):
        c = self.config
        b, s, d = x.shape
        h = d // c.num_heads
        z = self.attn_norm(x)
        q = self.q_proj(z).view(b, s, c.num_heads, h).transpose(1, 2)
        k = self.k_proj(z).view(b, s, c.num_kv_heads, h).transpose(1, 2)
        v = self.v_proj(z).view(b, s, c.num_kv_heads, h).transpose(1, 2)
        q, k = apply_rope(q, c.rope_base), apply_rope(k, c.rope_base)
        k = k.repeat_interleave(c.num_heads // c.num_kv_heads, dim=1)
        v = v.repeat_interleave(c.num_heads // c.num_kv_heads, dim=1)
        scores = q @ k.transpose(-2, -1) / math.sqrt(h)
        mask = torch.ones(s, s, dtype=torch.bool, device=x.device).triu(1)
        weights = scores.masked_fill(mask, -torch.inf).softmax(-1)
        x = x + self.o_proj((weights @ v).transpose(1, 2).reshape(b, s, d))
        z = self.ffn_norm(x)
        return x + self.down_proj(F.silu(self.gate_proj(z)) * self.up_proj(z))


class MiniLLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.d_model)
        self.layers = nn.ModuleList([DecoderLayer(config) for _ in range(config.num_layers)])
        self.norm = nn.RMSNorm(config.d_model, eps=config.eps)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight

    def forward(self, input_ids):
        if input_ids.ndim != 2 or not 0 < input_ids.shape[1] <= self.config.max_seq_len:
            raise ValueError('Expected nonempty (B, T) within max_seq_len')
        x = self.embed_tokens(input_ids)
        for layer in self.layers:
            x = layer(x)
        return self.lm_head(self.norm(x))


In [ ]:
import importlib
import mini_llm_reference
importlib.reload(mini_llm_reference)
MiniLLM = mini_llm_reference.MiniLLM


In [ ]:
with deterministic(0):
    model = MiniLLM(LLMConfig())
    print(model(torch.tensor([[1, 12, 25, 2]])).shape)
    print('Parameters:', sum(p.numel() for p in model.parameters()))

In [ ]:
check('mini_llm')